# 02 SVM baseline：手寫數字辨識

本 notebook 使用傳統機器學習建立 baseline。正式比較以原始 64 維資料為主；2D 降維資料僅用於視覺化決策邊界，以說明模型如何切分特徵空間。

> 說明：`load_digits` 是小型且乾淨的 toy dataset，因此 SVM RBF 取得高分屬合理現象。若要觀察真實影像任務中的限制，請接續參考 `03_svhn_svm_baseline.ipynb`。


In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay, classification_report
from sklearn.decomposition import PCA

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


## 1. 載入資料並固定切分

正式比較時，所有模型都要使用同一組 train/test split。

In [ ]:
digits = load_digits()
X = digits.data
y = digits.target
images = digits.images

X_train, X_test, y_train, y_test, img_train, img_test = train_test_split(
    X, y, images,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

print('X_train:', X_train.shape)
print('X_test:', X_test.shape)
print('y_train:', y_train.shape)
print('y_test:', y_test.shape)

## 2. 建立 SVM RBF baseline 與資料標準化

本節使用單一傳統機器學習模型作為 baseline：`SVM RBF`。

SVM RBF 會根據樣本之間的距離與 kernel similarity 建立分類邊界，因此特徵尺度會直接影響模型判斷。若某些特徵數值範圍較大，模型可能會過度受到這些特徵影響，而忽略其他尺度較小但仍有意義的特徵。

`StandardScaler()` 會將每個 feature 轉換成平均值約為 0、標準差約為 1 的尺度。這類前處理常見於以下情況：

補充比較：`/255` 或 `/16` 與 `StandardScaler` 不是同一件事。`/255` 是把一般 8-bit 圖片的像素壓到 0 到 1；`/16` 是把 `load_digits` 的 0 到 16 灰階值壓到 0 到 1。這類做法是針對整體像素範圍做正規化。`StandardScaler` 則是針對每一個像素位置，讓它在所有樣本中的平均值與變異量變得一致。神經網路處理影像時通常先用 `/255` 或 `/16` 就足夠；SVM 這類吃距離或 margin 的模型，仍建議加上 `StandardScaler`。

- 距離型模型：例如 KNN、SVM RBF，因為距離會被特徵尺度影響。
- 內積或 margin 型模型：例如 SVM、Logistic Regression，因為不同尺度會影響權重與決策邊界。
- 梯度最佳化模型：例如線性模型或神經網路，因為尺度差異過大會讓訓練較不穩定或收斂較慢。
- 使用 regularization 的模型：例如 L1 / L2 regularization，若特徵尺度不同，懲罰項對各特徵的影響會不公平。

若沒有標準化，可能出現以下問題：

- 大尺度特徵主導模型，其他特徵被弱化。
- SVM RBF 的距離計算失衡，`gamma` 的效果變得難以解讀。
- 模型對超參數更敏感，訓練結果不穩定。
- 訓練分數或測試分數下降，或需要花更多時間調參。

本節使用 `Pipeline` 將 `StandardScaler()` 與 `SVC(kernel='rbf')` 串在一起，確保訓練與測試流程一致。


In [ ]:
svm_rbf = Pipeline([
    ('scaler', StandardScaler()),
    ('model', SVC(kernel='rbf', C=1.0, gamma='scale', random_state=RANDOM_STATE))
])

start = time.perf_counter()
svm_rbf.fit(X_train, y_train)
elapsed = time.perf_counter() - start

train_pred = svm_rbf.predict(X_train)
test_pred = svm_rbf.predict(X_test)

results_df = pd.DataFrame([
    {
        'model': 'SVM RBF + StandardScaler',
        'train_acc': accuracy_score(y_train, train_pred),
        'test_acc': accuracy_score(y_test, test_pred),
        'training_time_sec': elapsed,
    }
])

fitted_models = {'SVM RBF': svm_rbf}
results_df


## 3. 看 SVM RBF 的混淆矩陣與報告

In [ ]:
best_name = 'SVM RBF'
best_model = fitted_models[best_name]
y_pred = best_model.predict(X_test)

print(classification_report(y_test, y_pred))

fig, ax = plt.subplots(figsize=(7, 7))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=ax, cmap='Blues', colorbar=False)
plt.title(f'{best_name} confusion matrix')
plt.show()


## 4. 看看分錯的圖片

分錯案例可補充 accuracy 以外的資訊，並顯示模型在筆跡形狀、位置偏移與類別相似度上的限制。


In [ ]:
wrong_idx = np.where(y_pred != y_test)[0]
print('number of wrong predictions:', len(wrong_idx))

n_show = min(12, len(wrong_idx))
fig, axes = plt.subplots(2, 6, figsize=(10, 4))
for ax, idx in zip(axes.ravel(), wrong_idx[:n_show]):
    ax.imshow(img_test[idx], cmap='gray_r')
    ax.set_title(f'true={y_test[idx]} pred={y_pred[idx]}')
    ax.axis('off')
for ax in axes.ravel()[n_show:]:
    ax.axis('off')
plt.tight_layout()
plt.show()

## 5. 2D 降維資料上的 SVM 決策邊界示範

本節為視覺化示範，用以呈現模型如何切分 2D 空間；此結果不納入正式公平比較。

這裡刻意使用 `PCA`，而不是 `t-SNE`，原因如下：

- `PCA` 是線性投影，訓練資料與測試資料可以使用同一個 `fit -> transform` 座標系，較適合示範「模型在 2D 特徵空間中如何切分邊界」。
- `t-SNE` 很適合觀察分群，通常會比 PCA 圖更有分群感，但它主要是視覺化工具，會刻意重排局部鄰近關係，2D 距離與邊界不適合解讀成原始特徵空間的真實分類規則。
- scikit-learn 的 `TSNE` 沒有像 PCA 一樣穩定的 `transform()` 可直接套到新測試資料，因此不適合作為本節 train/test 決策邊界示範的主要方法。

因此，本課程安排為：`01_digits_data_visualization.ipynb` 可用 t-SNE 看分群；本節用 PCA 產生 2D 特徵，再訓練 SVM 畫出決策邊界。


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_train_2d = pca.fit_transform(X_train_scaled)
X_test_2d = pca.transform(X_test_scaled)

svm_2d = SVC(kernel='rbf', C=5.0, gamma='scale', random_state=RANDOM_STATE)
svm_2d.fit(X_train_2d, y_train)
print('2D PCA SVM test accuracy (teaching demo only):', svm_2d.score(X_test_2d, y_test))

x_min, x_max = X_train_2d[:, 0].min() - 1, X_train_2d[:, 0].max() + 1
y_min, y_max = X_train_2d[:, 1].min() - 1, X_train_2d[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))
Z = svm_2d.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.figure(figsize=(9, 7))
plt.contourf(xx, yy, Z, alpha=0.25, cmap='tab10')
scatter = plt.scatter(X_train_2d[:, 0], X_train_2d[:, 1], c=y_train, cmap='tab10', s=18, edgecolor='k', linewidth=0.2)
plt.legend(*scatter.legend_elements(), title='Digits', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xlabel('PCA 1')
plt.ylabel('PCA 2')
plt.title('SVM decision regions on 2D PCA features (demo only)')
plt.tight_layout()
plt.show()

## 課後練習與思考題

### 練習 1：修改 SVM RBF 的 `C` 或 `gamma`

`C` 與 `gamma` 是 RBF SVM 中最常調整的兩個超參數，會直接影響模型複雜度與泛化能力。

- `C` 控制模型對訓練錯誤的容忍程度。
  - `C` 較小：允許一些訓練錯誤，決策邊界通常較平滑，可能比較不容易過擬合。
  - `C` 較大：更努力把訓練資料分對，決策邊界可能更貼近訓練資料，也比較容易過擬合。
- `gamma` 控制 RBF kernel 中每個樣本的影響範圍。
  - `gamma` 較小：單一樣本影響範圍較大，決策邊界較平滑。
  - `gamma` 較大：單一樣本影響範圍較小，決策邊界較細緻，但也可能過度貼合訓練資料。
- `gamma='scale'` 是 scikit-learn 的常用預設值，會依照特徵數量與資料變異量自動設定起始值。

練習時請比較不同 `C` 與 `gamma` 設定對 train accuracy 與 test accuracy 的影響。若 train accuracy 明顯提高，但 test accuracy 沒有同步提高，代表模型可能開始過擬合。

### 練習 2：找出 SVM 分錯的圖片

觀察前幾張錯誤案例，判斷錯誤是否和筆跡形狀、位置偏移或類別相似度有關。

### 練習 3：比較原始 64 維 SVM 與 2D PCA SVM

比較正式 64 維特徵與 2D PCA 特徵的 test accuracy，說明為何 2D PCA 決策邊界只適合視覺化，不列入正式公平比較。

### 思考題：為什麼本節不用 t-SNE 畫 SVM 決策邊界？

請說明 t-SNE 適合觀察分群，但不適合拿來解讀分類器在原始資料空間中的決策規則。


In [ ]:
# 練習 1 參考答案：修改 SVM RBF 的 C 或 gamma
# C 控制模型對訓練錯誤的懲罰強度；gamma 控制 RBF kernel 的影響範圍。
# 判讀時不要只看 test_acc，也要觀察 train_acc 與 test_acc 的落差。

configs = [
    {
        'C': 0.1,
        'gamma': 'scale',
        'note': '較小 C：容忍部分訓練錯誤，邊界通常較平滑。'
    },
    {
        'C': 1.0,
        'gamma': 'scale',
        'note': '中等 C：常作為基準設定。'
    },
    {
        'C': 10.0,
        'gamma': 'scale',
        'note': '較大 C：更努力分對訓練資料，需注意過擬合。'
    },
    {
        'C': 1.0,
        'gamma': 0.001,
        'note': '較小 gamma：樣本影響範圍較大，邊界較平滑。'
    },
    {
        'C': 1.0,
        'gamma': 0.01,
        'note': '較大 gamma：樣本影響範圍較小，邊界較細緻。'
    },
]

rows = []
for cfg in configs:
    model_params = {'C': cfg['C'], 'gamma': cfg['gamma']}
    model = Pipeline([
        ('scaler', StandardScaler()),
        ('model', SVC(kernel='rbf', random_state=RANDOM_STATE, **model_params))
    ])
    model.fit(X_train, y_train)
    train_acc = model.score(X_train, y_train)
    test_acc = model.score(X_test, y_test)
    rows.append({
        'C': cfg['C'],
        'gamma': cfg['gamma'],
        'train_acc': train_acc,
        'test_acc': test_acc,
        'train_test_gap': train_acc - test_acc,
        'note': cfg['note'],
    })

svm_param_df = pd.DataFrame(rows).sort_values('test_acc', ascending=False)
display(svm_param_df)



### 練習 1 參考答案：修改 SVM RBF 的 C 或 gamma

判讀重點：

1. `C` 變大時，模型通常會更努力把訓練資料分對，因此 `train_acc` 可能上升。
2. `gamma` 變大時，單一樣本只影響更小範圍，決策邊界可能更彎曲，也更容易貼近訓練資料。
3. 若 `train_test_gap` 變大，但 `test_acc` 沒有改善，就要懷疑模型正在過擬合。
4. `gamma='scale'` 是 scikit-learn 依據資料尺度自動設定的合理起點，實務上常先從它開始。


In [ ]:
# 練習 2 參考答案：找出 SVM 分錯的圖片
# 錯誤案例可用來觀察模型常見限制，例如筆跡太淡、形狀相似、位置偏移或圖片本身不清楚。

y_pred = best_model.predict(X_test)
wrong_idx = np.where(y_pred != y_test)[0]
print('wrong predictions:', len(wrong_idx))

n_show = min(12, len(wrong_idx))
fig, axes = plt.subplots(2, 6, figsize=(10, 4))
for ax, idx in zip(axes.ravel(), wrong_idx[:n_show]):
    ax.imshow(img_test[idx], cmap='gray_r')
    ax.set_title(f'true={y_test[idx]} pred={y_pred[idx]}')
    ax.axis('off')
for ax in axes.ravel()[n_show:]:
    ax.axis('off')
plt.tight_layout()
plt.show()


### 練習 2 參考答案：找出 SVM 分錯的圖片

錯誤案例應回到圖片本身觀察。若分錯圖片的筆跡形狀模糊、位置偏移明顯，或外觀看起來接近另一個數字，代表模型錯誤不只是分數問題，也反映資料本身的模糊性。


In [ ]:
# 練習 3 參考答案：比較原始 64 維 SVM 與 2D PCA SVM
# 64 維是正式模型輸入；2D PCA 是為了畫決策邊界而降維，會丟失大量資訊。

svm_64d = Pipeline([
    ('scaler', StandardScaler()),
    ('model', SVC(kernel='rbf', C=1.0, gamma='scale', random_state=RANDOM_STATE))
])
svm_64d.fit(X_train, y_train)
acc_64d = svm_64d.score(X_test, y_test)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_train_2d = pca.fit_transform(X_train_scaled)
X_test_2d = pca.transform(X_test_scaled)

svm_2d = SVC(kernel='rbf', C=5.0, gamma='scale', random_state=RANDOM_STATE)
svm_2d.fit(X_train_2d, y_train)
acc_2d = svm_2d.score(X_test_2d, y_test)

summary = pd.DataFrame([
    {'feature_space': 'original 64D', 'test_acc': acc_64d},
    {'feature_space': 'PCA 2D', 'test_acc': acc_2d},
])

display(summary)
print('PCA 2D explained variance:', pca.explained_variance_ratio_.sum())


### 練習 3 參考答案：比較原始 64 維 SVM 與 2D PCA SVM

2D PCA 適合視覺化決策邊界，但不適合作為正式模型比較的主要輸入。t-SNE 適合做分群視覺化，但不適合在本節拿來訓練 SVM 並解讀決策邊界，因為 t-SNE 的 2D 座標主要服務視覺化，距離與邊界不應被解讀為原始 64 維空間中的分類規則。
